# Line Emission Denoising — Sweep-Winner Validation + Artifact Diagnostics

Per the 2026-06-18 mentor pivot: full-image denoising of line-emission velocity channels
(NO patches), channel-by-channel, cube-level holdout, `bettermoments` M0/M1/M2 as the actual
scientific deliverable.

**What this run answers.** The 12-run random sweep (results committed:
`results/sweep_results.csv`) found a config at **PSNR 37.11 dB** — +4.16 dB over the V12
reference. But the beam A/B in the same session showed a pixel-metric win (+1.24 dB) that came
with **worse** M0 moment-map reliability (+59.8%±33.0% vs V12's +69.8%±15.2%). So a PSNR number
alone cannot be trusted as an improvement. This notebook:

1. **Retrains the sweep winner** (base=48, mult 1×2×4×8, lr=8.2e-4, alpha=0.888, no beam,
   batch 16) and runs it through the **same all-5-holdout moment-map protocol** V12 was judged
   on. That comparison decides whether the sweep winner becomes the new reference checkpoint.
2. **Quantifies the two known artifacts across every validation channel** — peak overshoot and
   low-SNR invented structure ("hallucination") — replacing the single channel-100 anecdote
   with statistics.

**Reference to beat — V12** (`unet_line_emission_continuum_best.pth`, fixed 30 epochs):
PSNR 32.95 dB | SSIM 0.9857 | MSE 0.000681; 5-cube holdout **M0 +69.8%±15.2% | M1 +17.5%±7.8%
| M2 +20.1%±14.3%** (all 5 cubes positive on all 3 moments).

Everything else is held identical to V12 and to the sweep: 14 cubes / 11 RunIDs, cube-level
split (3 RunID groups = 5 cubes held out, inference only), Gaussian channel sampling (center
100), 256×256 full images, continuum subtraction (n=5), shared **dirty**-scale per-channel
normalisation (invertible at inference), linear output head, seed=42.

**Kaggle setup:** GPU on, Internet on, `Add Input` → line-emission Dataset (FITS cubes).
The bootstrap locates it under `/kaggle/input/`.

**DDPM lives in `06-ddpm-line-emission.ipynb`** — separate notebook, never touches these files.

## 0. Bootstrap (clone repo for `src/`, locate data)

In [ ]:
import os, sys, subprocess, glob

ON_KAGGLE = os.path.exists('/kaggle')
if ON_KAGGLE:
    REPO='/kaggle/working/EXXA'; PKG=os.path.join(REPO,'DENOISING_DIFFUSION')
    if not os.path.exists(REPO):
        subprocess.run(['git','clone','--branch','line-emission','--depth','1','https://github.com/KrishanYadav333/EXXA.git',REPO], check=True)
    else:
        subprocess.run(['git','-C',REPO,'fetch','origin','line-emission'], check=True)
        subprocess.run(['git','-C',REPO,'reset','--hard','origin/line-emission'], check=True)
    # pytorch-msssim for the SSIM loss; bettermoments for moment-map evaluation (Sec. 6)
    subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','pytorch-msssim','bettermoments'], check=True)
    os.chdir(os.path.join(PKG,'notebooks'));  sys.path.insert(0, PKG)
    # find the line-emission data dir under /kaggle/input (contains run_* subfolders)
    # Fail here, loudly, if the dataset is not attached. Returning None instead
    # defers the error to the first os.path.join two cells later, where it reads as
    # "TypeError: expected str ... not NoneType" and says nothing about the cause.
    hits = (glob.glob('/kaggle/input/**/*_dirty.fits', recursive=True)
            or glob.glob('/kaggle/input/**/*dirty*.fits', recursive=True))
    if not hits:
        avail = sorted(glob.glob('/kaggle/input/*'))
        raise FileNotFoundError(
            'No *_dirty.fits found under /kaggle/input.\n'
            'The line-emission Dataset is probably not attached: '
            'Add Input -> Datasets -> your line-emission cubes.\n'
            f'Currently attached: {avail if avail else "(nothing)"}')
    DATA_DIR = os.path.dirname(os.path.dirname(hits[0]))
else:
    if os.path.basename(os.getcwd()) != 'notebooks' and os.path.isdir('notebooks'): os.chdir('notebooks')
    sys.path.insert(0, os.path.abspath('..'))
    DATA_DIR = '../data/Line Emission Data'
print('cwd:', os.getcwd(), '| DATA_DIR:', DATA_DIR)

## 0b. Pull latest code (re-run anytime — NO kernel restart needed)

After pushing new changes to the `line-emission` branch, re-run this cell to fetch them and
hot-reload the `src/` modules, then re-run the imports cell below.

In [ ]:
# Pull latest from the line-emission branch and hot-reload src/ (no kernel restart).
import os, sys, subprocess

ON_KAGGLE = os.path.exists('/kaggle')
REPO = '/kaggle/working/EXXA'

if ON_KAGGLE and os.path.exists(REPO):
    subprocess.run(['git', '-C', REPO, 'fetch', 'origin', 'line-emission'], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '--hard', 'origin/line-emission'], check=True)
    print(subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-1'],
                         capture_output=True, text=True).stdout.strip())
elif ON_KAGGLE:
    print('repo not cloned yet -- run the bootstrap cell (0.) first')

for _m in [m for m in list(sys.modules) if m == 'src' or m.startswith('src.')]:
    del sys.modules[_m]
print('src.* cleared from module cache -- now re-run the imports cell below.')

## 1. Imports, device, config

In [ ]:
import math, time
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from src.data.cube_split import split_cubes
from src.data.fits_cube_dataset import FITSChannelDataset, continuum_of
from src.models.unet import UNet
from src.training.sweep import train_unet, val_metrics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPU  = torch.cuda.device_count()
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print('device:', device, '| GPUs:', N_GPU,
      '->', [torch.cuda.get_device_name(i) for i in range(N_GPU)] if N_GPU else 'cpu')

OUT_DIR     = '../results'   # hoisted: section 4 writes its per-seed CSV here too
os.makedirs(OUT_DIR, exist_ok=True)

TARGET_SIZE = 256
N_SAMPLES   = 50      # identical to V12 and to the sweep -> numbers stay comparable
NW          = 4 if ON_KAGGLE else 0   # FITS I/O bottleneck; workers help on Kaggle

# V12 reference (fixed 30-epoch run) -- the bar this notebook is measured against
V12 = {'psnr': 32.95, 'ssim': 0.9857, 'mse': 0.000681,
       'M0': (69.8, 15.2), 'M1': (17.5, 7.8), 'M2': (20.1, 14.3)}
# Sweep winner as scored during the sweep (results/sweep_results.csv, run 7)
SWEEP_WINNER_PSNR = 37.109

## 2. Cube-level split (3 RunID groups held out for inference only)

In [ ]:
train_cubes, val_cubes, holdout_cubes = split_cubes(data_dir=DATA_DIR, n_holdout=3,
                                                     val_fraction=0.2, seed=SEED)

## 3. Datasets (full-image 256×256, continuum-subtracted, shared dirty-scale norm)

`return_beam=False` — the sweep winner does not use beam conditioning (beam correlated
*negatively* with PSNR across the 12 sweep runs, r=-0.33, and all top-5 configs had
`use_beam=False`).

In [ ]:
SUBTRACT_CONTINUUM = True
CONTINUUM_N        = 5
train_ds = FITSChannelDataset(train_cubes, n_samples=N_SAMPLES, target_size=TARGET_SIZE, seed=SEED,
                              subtract_continuum=SUBTRACT_CONTINUUM, continuum_n=CONTINUUM_N)
val_ds   = FITSChannelDataset(val_cubes,   n_samples=N_SAMPLES, target_size=TARGET_SIZE, seed=SEED,
                              subtract_continuum=SUBTRACT_CONTINUUM, continuum_n=CONTINUUM_N)
# D4-augmented view of the SAME cubes. Notebook 08 measured this as the best arm it
# tried -- +1.3 dB over the un-augmented winner, the best M0/M1/M2 of any arm, and the
# lowest invented-structure rate (29.7% vs 39.0%). With only 6 training disks, the eight
# lossless orientations are the cheapest way to add variety that costs no new data.
train_ds_aug = FITSChannelDataset(train_cubes, n_samples=N_SAMPLES, target_size=TARGET_SIZE,
                                  seed=SEED, subtract_continuum=SUBTRACT_CONTINUUM,
                                  continuum_n=CONTINUUM_N, augment=True)
print('train items:', len(train_ds), '| val items:', len(val_ds))

# Augmentation must vary the TRAIN item and leave VAL deterministic. Wired the wrong way
# round it silently makes validation non-reproducible, which reads as seed noise.
_n_orient = len({train_ds_aug[0][0].numpy().tobytes() for _ in range(40)})
_val_fixed = len({val_ds[0][0].numpy().tobytes() for _ in range(5)}) == 1
print(f'augmented item takes {_n_orient} orientations | val deterministic: {_val_fixed}')
assert _n_orient > 1 and _val_fixed, 'augmentation wiring is wrong -- stop and fix'

## 4. Train all three configurations across the same three seeds

The sweep's 37.109 dB came from run index 7, and `run_sweep` trains run *i* at `seed + i` —
so that number was produced at **seed 49**, while this notebook previously retrained at
`SEED = 42` and scored 30.28 / 29.92. That gap was written up as a reproduction failure; it
was a different draw. A single retrain at any one seed just samples the same distribution
again, so both configurations are trained at **seeds 42, 49 and 7** and reported as bands.

**V12 is retrained here rather than loaded.** No original V12 checkpoint survives anywhere
in the repository — only notebook 08's retrainings of the V12 *config*, and those used 1050
training items against this notebook's 350, so borrowing them would confound the comparison
with a 3× data difference. Retraining both configurations here holds data, split, schedule,
seeds and metric identical, which is the only way the comparison means anything.

The published V12 figures stay in the tables as a clearly labelled historical row: single
run, unmasked metric, no checkpoint available.

Six trainings, roughly 1.5–2 h. Rows append as each finishes, so a timeout costs one run.

### What earlier versions settled, and is therefore not re-run here

**The sweep (v15 §7-8).** A random hyperparameter search produced `WINNER` below. It is not
repeated: the search cost hours and its answer is a config, which is recorded. Its headline
`37.109 dB` came from run 7, whose seed was 49 — not this notebook's default 42 — which is
the whole of the "7 dB reproduction failure" that was chased for a session. Seed 49 is in
`SEEDS` for exactly that reason.

**Beam conditioning (v15 §4).** Feeding the per-cube beam `[sin2BPA, cos2BPA, bmaj, bmin]`
into the network is a **negative result** and is not offered as an arm. It won on pixels
and lost on science:

| | PSNR | M0 | M0 spread |
|---|---|---|---|
| beam-conditioned | 34.94 (**+1.24 dB**) | **+59.8%** | ±33.0 |
| V12 reference | 32.95 | +69.8% | ±15.2 |

A worse mean with double the variance, bought with a better pixel metric. That is the
pattern to watch for in every arm below, and it is why the verdict in section 6 keys on M0
and its spread rather than on PSNR.

In [ ]:
import csv, glob, shutil

# The ACTUAL V12 checkpoint, not a retrain of its config. Notebook 05 Version 12 wrote
# ../results/checkpoints/unet_line_emission_continuum_best.pth (epoch 28, val_loss 0.0034)
# inside /kaggle/working, so it is part of that version's Output: attach it as an Input and
# it is restored here. V12 trained on the same 350/100 items this notebook uses, so scoring
# it under the current metric is a like-for-like comparison, not an approximation.
V12_CKPT = os.path.join('../results/checkpoints', 'unet_line_emission_continuum_best.pth')
if ON_KAGGLE and not os.path.exists(V12_CKPT):
    _hits = sorted(glob.glob('/kaggle/input/**/unet_line_emission_continuum_best.pth',
                             recursive=True), key=os.path.getmtime)
    if _hits:
        os.makedirs(os.path.dirname(V12_CKPT), exist_ok=True)
        shutil.copy2(_hits[-1], V12_CKPT)
        print('restored the original V12 checkpoint from', _hits[-1])
HAVE_V12_CKPT = os.path.exists(V12_CKPT)
print('original V12 checkpoint available:', HAVE_V12_CKPT,
      '' if HAVE_V12_CKPT else '-- attach notebook 05 Version 12 Output to score it')

WINNER = dict(base_channels=48, channel_multipliers=(1, 2, 4, 8),
              lr=0.0008196504330730313, alpha=0.8877681051398497,
              sched_patience=8, use_beam=False, batch_size=16)
# V12's configuration, retrained here so it meets the winner on identical terms
V12_CFG = dict(base_channels=32, channel_multipliers=(1, 2, 4),
               lr=1e-3, alpha=0.8, sched_patience=5, use_beam=False, batch_size=16)

# 42 is this notebook's default; 49 is the seed that actually produced the sweep's 37.109
# (run index 7 -> base seed 42 + 7); 7 is a third, independent draw.
SEEDS = [42, 49, 7]
# Four arms, matching notebook 08 so the two are directly comparable:
#   sweep_winner      the sweep's config, the reference for the other arms
#   sweep_winner_aug  + D4 augmentation      -- does more variety help? (08: yes, +1.3 dB)
#   sweep_winner_p10  + patience=10          -- was V16's shortfall an early-stopping
#                                               artifact? (08: no -- same PSNR as _aug but
#                                               M0 +11.6 +/-58.7 against _aug's +38.7)
#   v12_cfg           V12's config retrained -- the baseline, now with an error bar
# The suffix selects the variation; the config dicts are otherwise identical, so each arm
# isolates exactly one change.
CONFIGS = {'sweep_winner': WINNER, 'sweep_winner_aug': WINNER,
           'sweep_winner_p10': dict(WINNER, patience=10), 'v12_cfg': V12_CFG}

CKPT_DIR = '../results/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
REPEAT_CSV = os.path.join(OUT_DIR, 'nb05_seed_repeats.csv')
FIELDS = ['config', 'seed', 'best_epoch', 'epochs_run', 'best_val_loss',
          'psnr', 'ssim', 'mse', 'wall_time_s']


def _done_rows(path):
    """Runs already on record, so a resumed session repeats none of them."""
    out = {}
    if not os.path.exists(path):
        return out
    with open(path, newline='') as f:
        for r in csv.DictReader(f):
            try:
                out[(r['config'], int(r['seed']))] = {k: r[k] for k in FIELDS}
            except (KeyError, ValueError):
                continue
    return out


done = _done_rows(REPEAT_CSV)
if done:
    print(f'[resume] {len(done)} run(s) already scored: {sorted(done)}')

rows = []
for name, cfg in CONFIGS.items():
    print(f'\n{"="*74}\n=== {name}: {cfg}\n{"="*74}', flush=True)
    for sd in SEEDS:
        ckpt = os.path.join(CKPT_DIR, f'nb05_{name}_seed{sd}.pth')
        if (name, sd) in done:
            r = done[(name, sd)]
            rows.append(r)
            print(f'--- {name} seed {sd}: SKIPPED, already done '
                  f'(PSNR {float(r["psnr"]):.4f}) ---', flush=True)
            continue
        print(f'\n--- {name}: seed {sd} ---', flush=True)
        _cfg = dict(cfg)
        _patience = _cfg.pop('patience', 5)   # p10 carries its own; everything else gets 5
        res = train_unet(train_ds_aug if name.endswith('_aug') else train_ds,
                         val_ds, device, **_cfg,
                         min_epochs=20, max_epochs=60, patience=_patience,
                         num_workers=NW, seed=sd, ckpt_path=ckpt, verbose=True)
        row = {'config': name, 'seed': sd,
               **{k: res[k] for k in FIELDS if k in res}}
        rows.append(row)
        new = not os.path.exists(REPEAT_CSV)
        with open(REPEAT_CSV, 'a', newline='') as f:
            w = csv.DictWriter(f, fieldnames=FIELDS)
            if new:
                w.writeheader()
            w.writerow({k: row.get(k, '') for k in FIELDS})
        print('  seed {}: PSNR {:.4f} | SSIM {:.4f} | best ep {} ({} run)'.format(
            sd, res['psnr'], res['ssim'], res['best_epoch'], res['epochs_run']))
        res.pop('model', None)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

band = {}
for name in CONFIGS:
    ps = [float(r['psnr']) for r in rows if r['config'] == name]
    band[name] = (float(np.mean(ps)), float(np.std(ps, ddof=1)) if len(ps) > 1 else 0.0, ps)

print(f'\n{"="*74}\nPSNR over seeds {SEEDS}\n{"="*74}')
for name, (m, s, ps) in band.items():
    print('  {:<14} {:.3f} +/- {:.3f}   ({})'.format(
        name, m, s, ', '.join(f'{p:.3f}' for p in ps)))
print('  {:<14} {:.3f}   <- sweep run 7, seed 49, single run'.format(
    'sweep row', SWEEP_WINNER_PSNR))
print('  {:<14} {:.3f}   <- published V12, single run, ORIGINAL UNMASKED metric'.format(
    'V12 (orig)', V12['psnr']))

_gap = band['sweep_winner'][0] - band['v12_cfg'][0]
_pool = float(np.sqrt(band['sweep_winner'][1]**2 + band['v12_cfg'][1]**2)) or 1e-12
print('\nsweep_winner - v12_cfg: {:+.3f} dB (pooled spread {:.3f})'.format(_gap, _pool))
print('  -> ' + ('within seed noise: INDISTINGUISHABLE' if abs(_gap) < _pool else
                 'exceeds the pooled spread: worth taking seriously at n=3'))
# the winner's own checkpoint used by later sections: best seed, named for compatibility
_best = max((r for r in rows if r['config'] == 'sweep_winner'),
            key=lambda r: float(r['psnr']))
CKPT_WINNER = os.path.join(CKPT_DIR, f'nb05_sweep_winner_seed{_best["seed"]}.pth')
print(f'\nbest sweep_winner seed: {_best["seed"]} (PSNR {float(_best["psnr"]):.4f}) -> {CKPT_WINNER}')


## 5. Loss curve

In [ ]:
tr_hist, va_hist = res['train_losses'], res['val_losses']
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(tr_hist)+1), tr_hist, marker='o', ms=3, label='train')
plt.plot(range(1, len(va_hist)+1), va_hist, marker='s', ms=3, label='val')
be = int(np.argmin(va_hist)) + 1
plt.axvline(be, color='gray', ls=':')
plt.scatter([be], [min(va_hist)], color='#E8715A', zorder=5, label=f'best ep {be}')
plt.xlabel('epoch'); plt.ylabel('HybridLoss')
plt.title('Sweep winner (base 48, 1x2x4x8, alpha {:.3f})'.format(WINNER['alpha']))
plt.legend(); plt.grid(alpha=0.3); plt.yscale('log'); plt.tight_layout()
os.makedirs('../results', exist_ok=True)
plt.savefig('../results/sweepwinner_loss.png', dpi=140); plt.show()

## 6. All-5-holdout moment maps — every checkpoint on ONE metric

All six checkpoints from section 4 are scored on the same five held-out cubes with the same
clipped + signal-masked metric. The comparison is only meaningful if nothing differs but the
model, so V12's configuration is evaluated here rather than quoting its published numbers.

The published V12 row is kept in the final table and labelled explicitly: it was produced by
a single run on the **original unmasked** metric, from a checkpoint that no longer exists.
It is context, not a comparison — the masked and unmasked metrics are not interchangeable
(the clip alone moves M1 by ~4×), so reading a delta between that row and any other would be
meaningless.


In [ ]:
import csv
import matplotlib.ticker as mticker
import bettermoments as bm
from astropy.io import fits
from src.evaluation.moment_maps import (generate_moment_maps,
                                        moment_improvement)

BS = 32
moments = ['M0', 'M1', 'M2']


def load_net(path):
    ck = torch.load(path, map_location=device, weights_only=False)
    net = UNet(in_channels=1, out_channels=1, base_channels=ck['base_channels'],
               channel_multipliers=ck['channel_multipliers'], time_emb_dim=128,
               num_res_blocks=2, groups=math.gcd(8, ck['base_channels']),
               beam_dim=ck.get('beam_dim', 0)).to(device)
    net.load_state_dict(ck['model_state_dict']); net.eval()
    return net, ck


def mdiff(a, b):
    mask = np.isfinite(a) & np.isfinite(b)
    return float(np.nanmean(np.abs(a[mask] - b[mask])))


def denoise_cube(ho_entry, net):
    """Continuum-subtract, per-channel dirty-scale norm, denoise, invert. Returns arrays."""
    with fits.open(ho_entry['dirty'], memmap=False) as hdul:
        dirty_raw = np.ascontiguousarray(hdul[0].data).astype(np.float32)
    C, H, W = dirty_raw.shape
    dirty_csub = dirty_raw - continuum_of(dirty_raw, CONTINUUM_N)[None, :, :]
    del dirty_raw
    los  = dirty_csub.reshape(C, -1).min(axis=1)
    his  = dirty_csub.reshape(C, -1).max(axis=1)
    rngs = his - los
    norm = np.zeros_like(dirty_csub)
    nz = rngs > 0
    norm[nz] = (dirty_csub[nz] - los[nz, None, None]) / rngs[nz, None, None]
    out = np.empty_like(dirty_csub)
    with torch.no_grad():
        for s in range(0, C, BS):
            t = torch.from_numpy(norm[s:s+BS])[:, None].float().to(device)
            t256 = F.interpolate(t, (TARGET_SIZE, TARGET_SIZE), mode='bilinear', align_corners=False)
            tz = torch.zeros(t256.size(0), dtype=torch.long, device=device)
            pred = net(t256, tz)
            back = F.interpolate(pred, (H, W), mode='bilinear', align_corners=False)[:, 0].cpu().numpy()
            for k in range(back.shape[0]):
                ch = s + k
                out[ch] = back[k] * rngs[ch] + los[ch] if rngs[ch] > 0 else \
                          np.full((H, W), los[ch], np.float32)
    del norm
    return out, dirty_csub


# clean/dirty moment maps once per cube -- identical for every checkpoint scored below
cache = {}
for ho in holdout_cubes:
    with fits.open(ho['clean'], memmap=False) as h:
        ccsub = np.ascontiguousarray(h[0].data).astype(np.float32)
    ccsub -= continuum_of(ccsub, CONTINUUM_N)[None]
    _cube, velax = bm.load_cube(ho['dirty'])
    del _cube
    with fits.open(ho['dirty'], memmap=False) as h:
        dcsub = np.ascontiguousarray(h[0].data).astype(np.float32)
    dcsub -= continuum_of(dcsub, CONTINUUM_N)[None]
    cache[ho['folder']] = {'velax': velax,
                           'clean': generate_moment_maps(None, data_velax=(ccsub, velax)),
                           'dirty': generate_moment_maps(None, data_velax=(dcsub, velax))}
    del ccsub, dcsub
print('cached clean/dirty moment maps for', len(cache), 'cubes')

MOM_CSV = os.path.join(OUT_DIR, 'nb05_moment_by_seed.csv')
MFIELDS = (['config', 'seed', 'cube'] +
           [f'imp_{m}' for m in moments] + [f'imp_{m}_all' for m in moments])


def _done_moments(path):
    out = {}
    if os.path.exists(path):
        with open(path, newline='') as f:
            for r in csv.DictReader(f):
                try:
                    out[(r['config'], int(r['seed']), r['cube'])] = r
                except (KeyError, ValueError):
                    continue
    return out


mdone = _done_moments(MOM_CSV)
if mdone:
    print(f'[resume] {len(mdone)} cube-score(s) already on record')

per_run = {}
for name in CONFIGS:
    for sd in SEEDS:
        ck_path = os.path.join(CKPT_DIR, f'nb05_{name}_seed{sd}.pth')
        if not os.path.exists(ck_path):
            print(f'  !! missing checkpoint {os.path.basename(ck_path)} -- section 4 incomplete')
            continue
        net = None
        rws = []
        print(f'\n=== {name} seed {sd} ===', flush=True)
        for ho in holdout_cubes:
            key = (name, sd, ho['folder'])
            if key in mdone:
                r = mdone[key]
                rws.append(r)
                print('  {:<24} M0 {:>7.1f}%  M1 {:>7.1f}%  M2 {:>7.1f}%  (already scored)'.format(
                    ho['folder'], float(r['imp_M0']), float(r['imp_M1']), float(r['imp_M2'])))
                continue
            if net is None:
                net, _ck = load_net(ck_path)
            den, _ = denoise_cube(ho, net)
            e = cache[ho['folder']]
            no = generate_moment_maps(None, data_velax=(den, e['velax']))
            del den
            imp = moment_improvement(e['clean'], e['dirty'], no)
            r = {'config': name, 'seed': sd, 'cube': ho['folder'],
                 **{f'imp_{m}': round(imp[m], 2) for m in moments},
                 **{f'imp_{m}_all': round(imp[f'{m}_all'], 2) for m in moments}}
            rws.append(r)
            new = not os.path.exists(MOM_CSV)
            with open(MOM_CSV, 'a', newline='') as f:
                w = csv.DictWriter(f, fieldnames=MFIELDS)
                if new:
                    w.writeheader()
                w.writerow({k: r.get(k, '') for k in MFIELDS})
            print('  {:<24} M0 {:>7.1f}%  M1 {:>7.1f}%  M2 {:>7.1f}%'.format(
                ho['folder'], r['imp_M0'], r['imp_M1'], r['imp_M2']))
        per_run[(name, sd)] = rws
        if net is not None:
            del net
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

# The original V12 checkpoint, scored on exactly the same cubes and metric.
v12_orig = None
if HAVE_V12_CKPT:
    print('\n=== V12 ORIGINAL checkpoint ===', flush=True)
    _net, _ck = load_net(V12_CKPT)
    print(f'  epoch {_ck.get("epoch")} | val_loss {float(_ck.get("val_loss", float("nan"))):.4f}')
    _rw = []
    for ho in holdout_cubes:
        den, _ = denoise_cube(ho, _net)
        e = cache[ho['folder']]
        no = generate_moment_maps(None, data_velax=(den, e['velax']))
        del den
        imp = moment_improvement(e['clean'], e['dirty'], no)
        _rw.append(imp)
        print('  {:<24} M0 {:>7.1f}%  M1 {:>7.1f}%  M2 {:>7.1f}%'.format(
            ho['folder'], imp['M0'], imp['M1'], imp['M2']))
    v12_orig = {m: (float(np.mean([r[m] for r in _rw])),
                    float(np.std([r[m] for r in _rw], ddof=1))) for m in moments}
    print('  -> ' + '  '.join('{} {:+.1f}%+/-{:.1f}'.format(m, *v12_orig[m]) for m in moments))
    del _net
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# per-config band: mean over cubes for each seed, then spread ACROSS seeds
band_mom = {}
for name in CONFIGS:
    per_seed = {m: [] for m in moments}
    for sd in SEEDS:
        rws = per_run.get((name, sd))
        if not rws:
            continue
        for m in moments:
            per_seed[m].append(float(np.mean([float(r[f'imp_{m}']) for r in rws])))
    band_mom[name] = {m: (float(np.mean(per_seed[m])),
                          float(np.std(per_seed[m], ddof=1)) if len(per_seed[m]) > 1 else 0.0)
                      for m in moments if per_seed[m]}

print('\n' + '=' * 96)
print('MOMENT-MAP COMPARISON -- all rows below the line share ONE metric (clipped + signal-masked)')
print('=' * 96)
print('{:<34}{:>10}{:>18}{:>18}{:>18}'.format('', 'PSNR', 'M0 (%)', 'M1 (%)', 'M2 (%)'))
print('-' * 96)
for name, label in (('sweep_winner', 'sweep winner (3 seeds)'),
                    ('sweep_winner_aug', 'sweep winner + D4 aug (3 seeds)'),
                    ('sweep_winner_p10', 'sweep winner + patience 10 (3 seeds)'),
                    ('v12_cfg', 'V12 config (3 seeds)')):
    if name not in band_mom:
        continue
    b = band_mom[name]
    pm, ps, _ = band[name]
    print('{:<34}{:>7.2f}{:>3}'.format(label, pm, '') +
          ''.join('{:>11.1f}+/-{:<5.1f}'.format(b[m][0], b[m][1]) for m in moments))
if v12_orig:
    print('{:<34}{:>7}{:>3}'.format('V12 ORIGINAL ckpt (1 model)', '--', '') +
          ''.join('{:>11.1f}+/-{:<5.1f}'.format(*v12_orig[m]) for m in moments))
    print('{:<34}'.format('  (cube-to-cube spread, n=5)'))
print('-' * 96)
print('{:<34}{:>7.2f}{:>3}'.format('V12 published (ORIGINAL run,', V12['psnr'], '') +
      ''.join('{:>11.1f}+/-{:<5.1f}'.format(*V12[m]) for m in moments))
print('{:<34}'.format('  UNMASKED metric -- NOT comparable)'))
print('=' * 96)
print('Rows above the second line share one metric and may be compared. The final row is the')
print('published V12 figure on the ORIGINAL unmasked metric, kept for provenance only: the')
print('clip alone moves M1 by roughly 4x, so a delta against it measures the metric change,')
print('not the models. Note its spread is cube-to-cube where the 3-seed rows are across seeds.')

if 'sweep_winner' in band_mom and 'v12_cfg' in band_mom:
    print('\nSweep winner vs V12 config, same metric, same seeds:')
    for m in moments:
        gm, gs = band_mom['sweep_winner'][m]
        vm, vs = band_mom['v12_cfg'][m]
        gap = gm - vm
        pooled = float(np.sqrt(gs**2 + vs**2)) or 1e-12
        verdict = ('within seed spread -> INDISTINGUISHABLE' if abs(gap) < pooled else
                   'exceeds the seed spread -> real at n=3' if abs(gap) > 2*pooled else
                   'suggestive, not established at n=3')
        print('  {}: {:+6.1f} pp  (seed spread {:.1f}) -> {}'.format(m, gap, pooled, verdict))
    print('\nn=3 seeds; the std is itself a noisy estimate. This is a guard against')
    print('over-claiming, not a significance test.')

# Augmentation is the one change here that is cleanly attributable: identical config,
# identical seeds, identical data -- only the training view differs.
if 'sweep_winner' in band_mom and 'sweep_winner_aug' in band_mom:
    print('\nEffect of D4 augmentation (same config, same seeds):')
    _dp = band['sweep_winner_aug'][0] - band['sweep_winner'][0]
    _pp = float(np.sqrt(band['sweep_winner_aug'][1]**2 + band['sweep_winner'][1]**2)) or 1e-12
    print('  PSNR: {:+.2f} dB  (pooled seed spread {:.2f}) -> {}'.format(
        _dp, _pp, 'within seed noise' if abs(_dp) < _pp else 'exceeds the seed spread'))
    for m in moments:
        am, asd = band_mom['sweep_winner_aug'][m]
        wm, wsd = band_mom['sweep_winner'][m]
        gap = am - wm
        pooled = float(np.sqrt(asd**2 + wsd**2)) or 1e-12
        print('  {}: {:+6.1f} pp  (seed spread {:.1f}) -> {}'.format(
            m, gap, pooled,
            'within seed spread' if abs(gap) < pooled else
            'exceeds the seed spread -> real at n=3' if abs(gap) > 2*pooled else
            'suggestive, not established at n=3'))
    print('  Notebook 08 measured +1.3 dB and the lowest invented-structure rate for this')
    print('  arm on a different split; agreement across both is worth more than either.')


# Sections 7 and 8 diagnose ONE model. Use the best sweep-winner seed, and say which --
# an artifact rate quoted without naming its checkpoint is not reproducible.
# Best CHECKPOINT across every arm, not a hardcoded arm: if augmentation wins there is no
# reason to diagnose the model it beat. The arm is printed, because an artifact rate quoted
# without naming its checkpoint is not reproducible.
_best_row = max(rows, key=lambda r: float(r['psnr']))
BEST_CONFIG = _best_row['config']
BEST_SEED = int(_best_row['seed'])
CKPT_WINNER = os.path.join(CKPT_DIR, f'nb05_{BEST_CONFIG}_seed{BEST_SEED}.pth')
eval_net, _eval_ck = load_net(CKPT_WINNER)
print(f'\nsections 7-8 diagnose {BEST_CONFIG} seed {BEST_SEED} '
      f'(PSNR {float(_best_row["psnr"]):.4f}, epoch {_eval_ck.get("epoch")})')


## 6b. Moment-map figures — the disks, on a scale that shows them

The numbers above say the denoiser helps. These say *how*, and they are what a reader
actually looks at.

Off the source there is no line, so M1 and M2 are fits to noise and take values spanning
tens of km/s. Plotted unmasked they set the colour limits and the disk itself gets a few
percent of the ramp — which is why earlier moment figures looked like static. Here M1/M2
are shown over the same signal mask the scores use, every column shares one scale across
dirty/denoised/clean, and velocity is centred so red/blue means receding/approaching.

In [ ]:
from src.evaluation.moment_maps import plot_moment_comparison

# One figure per holdout cube for the diagnosed checkpoint. `cache` already holds the
# clean/dirty maps; only the denoised pass is new.
FIG_DIR = os.path.join(OUT_DIR, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

moment_figs = []
for ho in holdout_cubes:
    e = cache[ho['folder']]
    den, _ = denoise_cube(ho, eval_net)
    no = generate_moment_maps(None, data_velax=(den, e['velax']))
    imp = moment_improvement(e['clean'], e['dirty'], no)
    tag = '{}  —  {} seed {}'.format(ho['folder'], BEST_CONFIG, BEST_SEED)
    path = os.path.join(FIG_DIR, 'moments_{}.png'.format(ho['folder']))
    fig = plot_moment_comparison(e['clean'], e['dirty'], no, save_path=path, tag=tag)
    fig.text(0.5, -0.012,
             'M0 {:+.1f}%   M1 {:+.1f}%   M2 {:+.1f}%   (improvement over dirty, '
             'scored on {:,} masked px)'.format(imp['M0'], imp['M1'], imp['M2'], imp['n_px']),
             ha='center', fontsize=11)
    plt.show()
    moment_figs.append(path)
    print('  {:<28} M0 {:+6.1f}  M1 {:+6.1f}  M2 {:+6.1f}  -> {}'.format(
        ho['folder'], imp['M0'], imp['M1'], imp['M2'], os.path.basename(path)))
    del den, no

print('\nsaved {} moment-map figures -> {}'.format(len(moment_figs), FIG_DIR))

## 6c. Validation channels — dirty | denoised | clean

Restored from Version 12, which is the last version that had it: v15 onward dropped the
qualitative view entirely and reported only numbers. A moment score says the denoiser is
closer to truth on average; it cannot show *how* it fails, and section 8's worst-offender
panel only ever shows the extremes. These are ordinary channels, chosen by seed.

In [ ]:
import random

_idxs = random.Random(SEED).sample(range(len(val_ds)), 5)
fig, ax = plt.subplots(5, 3, figsize=(10, 16))
for k, t in enumerate(['dirty', 'U-Net denoised', 'clean GT']):
    ax[0, k].set_title(t, fontweight='bold')
with torch.no_grad():
    for r, ix in enumerate(_idxs):
        d, c = val_ds[ix]
        pred = eval_net(d[None].to(device),
                        torch.zeros(1, dtype=torch.long, device=device))[0, 0].cpu().numpy()
        cl = c[0].numpy()
        # Floor-relative window, as section 8 uses. Under shared dirty-scale normalisation
        # the background sits near +0.32, not 0, so vmin=0 spends a third of the ramp below
        # any real pixel and every panel washes out.
        _lo, _hi = float(np.percentile(cl, 1)), float(cl.max())
        ci, ch = val_ds.index[ix]
        for col, im in enumerate([d[0].numpy(), pred, cl]):
            ax[r, col].imshow(im, cmap='inferno', vmin=_lo, vmax=_hi)
            ax[r, col].axis('off')
        ax[r, 0].text(0.0, 1.02, '{} ch {}'.format(val_ds.cube_paths[ci][2], ch),
                      transform=ax[r, 0].transAxes, fontsize=8, va='bottom', ha='left')
fig.suptitle('Line-emission U-Net — validation channels ({} seed {})'.format(
    BEST_CONFIG, BEST_SEED), fontweight='bold', y=0.995)
plt.tight_layout()
os.makedirs('../experiments', exist_ok=True)
plt.savefig('../experiments/line_emission_unet_comparison.png', dpi=140, bbox_inches='tight')
plt.show()
print('saved -> experiments/line_emission_unet_comparison.png')

## 6d. Every arm vs V12 — the headline figure

v16 and v17 drew this as two bars, winner against V12. There are three arms now, so it
generalises to one bar per arm plus V12, with the per-cube points overlaid: a mean that
rests on one good cube and four mediocre ones should not look like a mean that does not.

Read the error bars with care. The arm bars are the spread **across seeds**; V12's is the
spread **across cubes**, because that is all the published figure recorded. They are not
the same quantity and the figure says so.

In [ ]:
import matplotlib.ticker as mticker

_arms = [n for n in ('sweep_winner', 'sweep_winner_aug', 'sweep_winner_p10', 'v12_cfg')
         if n in band_mom]
_labels = {'sweep_winner': 'sweep winner', 'sweep_winner_aug': 'winner + D4 aug',
           'sweep_winner_p10': 'winner + patience 10', 'v12_cfg': 'V12 config (retrained)'}
_colors = {'sweep_winner': '#4E91C7', 'sweep_winner_aug': '#5FA860',
           'sweep_winner_p10': '#8A6FB0', 'v12_cfg': '#C77E4E'}

fig, ax = plt.subplots(figsize=(11, 5.5))
x = np.arange(len(moments))
w_ = 0.8 / (len(_arms) + 1)
for k, name in enumerate(_arms):
    off = (k - len(_arms) / 2) * w_
    ax.bar(x + off, [band_mom[name][m][0] for m in moments], w_,
           yerr=[band_mom[name][m][1] for m in moments], capsize=4,
           label=_labels.get(name, name), color=_colors.get(name, '#888888'), alpha=0.9)
    for i, m in enumerate(moments):
        pts = [float(r[f'imp_{m}']) for sd in SEEDS for r in per_run.get((name, sd), [])]
        if pts:
            ax.scatter([i + off] * len(pts), pts, color='k', s=12, zorder=5, alpha=0.55)
# V12 published, on the ORIGINAL unmasked metric -- hatched so it cannot be misread as
# one of the like-for-like bars beside it.
off = (len(_arms) - len(_arms) / 2) * w_
ax.bar(x + off, [V12[m][0] for m in moments], w_, yerr=[V12[m][1] for m in moments],
       capsize=4, label='V12 published (unmasked metric)', color='#9B9B9B', alpha=0.9,
       hatch='//')
ax.axhline(0, color='#333333', lw=0.8, ls='--')
ax.set_xticks(x)
ax.set_xticklabels(['Moment 0\n(intensity)', 'Moment 1\n(velocity)', 'Moment 2\n(dispersion)'])
ax.set_ylabel('Improvement over dirty (%)')
ax.set_title('Moment-map improvement, 5 held-out cubes\n'
             'coloured bars: spread across {} seeds  |  hatched: V12, spread across cubes'
             .format(len(SEEDS)))
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%+.0f%%'))
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'moment_all_arms_vs_v12.png'), dpi=140, bbox_inches='tight')
plt.show()
print('saved -> results/moment_all_arms_vs_v12.png')

## 7. Artifact diagnostics — peak overshoot + low-SNR invented structure

Both known artifacts have so far been documented from a **single** channel (channel 100), which
the V7/V9 variance lesson says is not enough. This measures them across **every** validation
channel:

- **Peak overshoot:** `denoised.max() / clean.max()` per channel. Prior single-channel figure
  was 1.151 (≈15% too bright). Reported here as a distribution.
- **Invented structure ("hallucination"):** for each channel, look only at pixels where clean is
  background (`clean < 10% of clean.max()`) and count how many the model pushed above 20% of
  clean's peak — signal asserted where the ground truth has none. Reported as an area fraction
  plus a connected-component count (how many distinct fake blobs), split by channel SNR so we
  can say *whether it is a low-SNR-specific failure*, which is the open question flagged to the
  mentor.

This is characterisation, not a fix — but it turns "hallucination observed once" into a number
the midterm report can state and later runs can be compared against.

In [ ]:
from src.evaluation.artifacts import channel_artifacts, summarise, INVENT_FRAC, BLOB_MIN_PX

eval_net.eval()
rows = []
with torch.no_grad():
    for ix in range(len(val_ds)):
        d, c = val_ds[ix]
        pred = eval_net(d[None].to(device),
                        torch.zeros(1, dtype=torch.long, device=device))[0, 0].cpu().numpy()
        cl, dt = c[0].numpy(), d[0].numpy()
        if float(cl.max()) <= 0:
            continue                      # empty channel: nothing to score against
        ci, ch = val_ds.index[ix]
        rows.append({'cube': val_ds.cube_paths[ci][2], 'channel': int(ch),
                     **channel_artifacts(cl, dt, pred)})

s = summarise(rows)
print('validation channels analysed:', s['n_channels'])
print('\nPEAK OVERSHOOT  denoised.max / clean.max   (1.0 = perfect; prior ch-100 figure 1.151)')
print('  mean {overshoot_mean:.3f} | median {overshoot_median:.3f} | p90 {overshoot_p90:.3f} | max {overshoot_max:.3f}'.format(**s))
print('  channels overshooting by >10%: {:.0f}%'.format(100 * s['frac_over_10pct']))
print('\nNEGATIVE FLOOR LEAK  denoised.min  (clean floor ~0; prior figure -0.0017)')
print('  mean {floor_leak_mean:+.5f} | most negative {floor_leak_min:+.5f}'.format(**s))
print('\nINVENTED STRUCTURE  (background pixels pushed above {:.0%} of clean peak,'
      ' blobs >= {} px)'.format(INVENT_FRAC, BLOB_MIN_PX))
print('  channels with >=1 fake blob: {:.0f}% | blobs/channel {:.2f}'.format(
    100 * s['frac_channels_with_blob'], s['blobs_per_channel']))
print('  mean invented area fraction of background: {invented_frac_mean:.4%} |'
      ' worst channel: {invented_frac_max:.4%}'.format(**s))

if 'snr_median' in s:
    print('\n  SNR split at median {snr_median:.1f}  ->  is invented structure low-SNR specific?'.format(**s))
    print('    low-SNR  half: blobs/channel {low_snr_blobs_per_channel:.2f} |'
          ' invented area {low_snr_invented_frac:.4%} | overshoot {low_snr_overshoot:.3f}'.format(**s))
    print('    high-SNR half: blobs/channel {high_snr_blobs_per_channel:.2f} |'
          ' invented area {high_snr_invented_frac:.4%} | overshoot {high_snr_overshoot:.3f}'.format(**s))

ov   = np.array([r['overshoot'] for r in rows])
leak = np.array([r['floor_leak'] for r in rows])
inv   = np.array([r['invented_frac'] for r in rows])
snr  = np.array([r['snr'] for r in rows])

diag_csv = os.path.join(OUT_DIR, 'artifact_diagnostics_sweepwinner.csv')
with open(diag_csv, 'w', newline='') as cf:
    w = csv.DictWriter(cf, fieldnames=['cube', 'channel', 'snr', 'overshoot', 'floor_leak',
                                       'invented_frac', 'invented_blobs'])
    w.writeheader()
    for r in rows: w.writerow(r)
print('\nper-channel diagnostics saved ->', diag_csv)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(ov, bins=25, color='#4E91C7', alpha=0.85)
ax[0].axvline(1.0, color='k', ls='--', label='no overshoot')
ax[0].axvline(1.151, color='#E8715A', ls=':', label='prior ch-100 figure')
ax[0].set_xlabel('denoised.max / clean.max'); ax[0].set_ylabel('channels')
ax[0].set_title('Peak overshoot'); ax[0].legend(fontsize=8)
ax[1].scatter(snr, inv * 100, s=18, alpha=0.7, color='#4E91C7')
ax[1].set_xlabel('channel SNR (clean peak / background std)')
ax[1].set_ylabel('invented background area (%)')
ax[1].set_title('Invented structure vs SNR'); ax[1].set_xscale('log')
ax[2].hist(leak, bins=25, color='#9B9B9B', alpha=0.9)
ax[2].axvline(0.0, color='k', ls='--')
ax[2].set_xlabel('denoised.min'); ax[2].set_title('Negative floor leak')
for a in ax: a.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'artifact_diagnostics_sweepwinner.png'), dpi=140); plt.show()

## 8. Worst offenders — visual check of the invented structure

The three validation channels with the largest invented-area fraction, shown dirty / denoised /
clean. If the diagnostic in Section 7 is measuring something real, the fake structure should be
visible here; if these look fine, the thresholds need revisiting before the number is quoted.

In [ ]:
# Rank by invented structure, then show the worst channels.
#
# Four things were wrong here and all trace to the same assumption -- that the clean
# background sits near zero, which is false under shared dirty-scale normalisation:
#
#   1. every invented_frac was 0.0 (empty background mask), so the sort was a no-op
#      and this simply showed the FIRST three channels -- which is why all three came
#      from one cube and were labelled "worst".
#   2. SNR printed as nan for the same reason.
#   3. the row label was drawn with set_title on the same axes that already carried
#      the 'dirty' column header, so the two strings overlapped.
#   4. the colour scale ran vmin=0..clean.max(), but the data floor is ~+0.32, so a
#      third of the ramp was spent below any real pixel and every panel washed out.
#
# Ranking now breaks ties on overshoot and spreads picks across cubes, so a degenerate
# metric can never again masquerade as a considered selection.
def _rank(r):
    return (-r['invented_frac'], -r.get('invented_blobs', 0), -abs(r['overshoot'] - 1.0))

ordered = sorted(rows, key=_rank)
if all(r['invented_frac'] == 0 for r in rows):
    print('NOTE: no channel scored any invented structure -- showing the largest '
          'overshoot instead, and check n_background_px before trusting this.')

seen_cubes, worst = set(), []
for r in ordered:                      # prefer distinct cubes before repeating one
    if r['cube'] not in seen_cubes:
        worst.append(r); seen_cubes.add(r['cube'])
    if len(worst) == 3:
        break
for r in ordered:
    if len(worst) == 3:
        break
    if r not in worst:
        worst.append(r)

key = {(r['cube'], r['channel']) for r in worst}
picks = [ix for ix in range(len(val_ds))
         if (val_ds.cube_paths[val_ds.index[ix][0]][2], int(val_ds.index[ix][1])) in key][:3]

fig, ax = plt.subplots(len(picks), 3, figsize=(11, 3.9 * len(picks)), squeeze=False)
with torch.no_grad():
    for r, ix in enumerate(picks):
        d, c = val_ds[ix]
        pred = eval_net(d[None].to(device),
                        torch.zeros(1, dtype=torch.long, device=device))[0, 0].cpu().numpy()
        cl = c[0].numpy()
        # floor-relative window: the background is NOT at 0, so anchoring vmin there
        # throws away most of the colour range
        floor = float(np.percentile(cl, 1)); vmax = float(cl.max())
        ci, ch = val_ds.index[ix]
        info = next(q for q in rows
                    if q['cube'] == val_ds.cube_paths[ci][2] and q['channel'] == int(ch))
        for col, im in enumerate([d[0].numpy(), pred, cl]):
            ax[r][col].imshow(im, cmap='inferno', vmin=floor, vmax=vmax)
            ax[r][col].axis('off')
            if r == 0:      # column headers on the top row only
                ax[r][col].set_title(['dirty', 'denoised', 'clean GT'][col], fontweight='bold')
        snr = info['snr']
        snr_s = f'{snr:.1f}' if np.isfinite(snr) else 'n/a'
        # row label as figure text to the LEFT of the axes -- never a title, so it
        # cannot collide with the column headers
        ax[r][0].text(0.0, 1.02,
                      '{} ch {}   SNR {} | invented {:.3%} | {} blob(s) | bg {} px'.format(
                          val_ds.cube_paths[ci][2], ch, snr_s, info['invented_frac'],
                          info['invented_blobs'], info.get('n_background_px', '?')),
                      transform=ax[r][0].transAxes, fontsize=8, va='bottom', ha='left')

fig.suptitle('Worst invented-structure channels (colour scale: clean floor to clean peak)',
             fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.98])
os.makedirs('../experiments', exist_ok=True)
plt.savefig('../experiments/hallucination_worst_channels.png', dpi=140, bbox_inches='tight')
plt.show()
print('saved -> experiments/hallucination_worst_channels.png')

## 9. Persist artifacts to /kaggle/working (survives session end)

Copies the checkpoint and every CSV to the top level of `/kaggle/working` (outside the git
clone) so `kaggle kernels output <slug>` retrieves them. **The CSVs must be pulled down and
committed to `results/` manually** — Kaggle's GitHub sync pushes only the `.ipynb`, not files
written during the session.

In [ ]:
import glob

# RULES.md #1: the per-model saves already happened in section 4, as each model finished.
# This is the backstop for everything else, and for a session that resumed from CSVs
# without retraining.
if ON_KAGGLE:
    n_ck = n_res = 0
    for _src in sorted(glob.glob(os.path.join(CKPT_DIR, '*.pth'))):
        _dst = os.path.join('/kaggle/working', os.path.basename(_src))
        if not os.path.exists(_dst):
            shutil.copy2(_src, _dst)
        n_ck += 1
    for _dir in (OUT_DIR, '../experiments'):
        for _pat in ('*.csv', '*.png', '*.npz'):
            for _src in sorted(glob.glob(os.path.join(_dir, _pat))):
                shutil.copy2(_src, os.path.join('/kaggle/working', os.path.basename(_src)))
                n_res += 1
    print('persisted {} checkpoint(s) and {} result file(s) to /kaggle/working'.format(
        n_ck, n_res))
    if n_ck == 0:
        print('  WARNING: no checkpoints in ' + CKPT_DIR)
    print('\nTo resume: Add Input -> Notebooks -> THIS notebook. Do NOT re-upload a .pth as')
    print('a Dataset -- Kaggle unpacks it and torch.load then fails with "Is a directory".')
else:
    print('not on Kaggle -- nothing to persist')

## Collect this notebook's outputs

Every artifact this notebook produced, copied into a **versioned run folder** together with
a manifest recording the commit, the UTC time and a checksum per file:

    <outputs>/05-unet-line-emission/<UTC timestamp>_<git sha>/

Nothing overwrites anything, so two runs can be compared directly. Results used to land in
one flat `results/` directory with no record of which notebook or which revision wrote them
-- a moment CSV from before the M2 noise-clip fix was indistinguishable from one after it.

On Kaggle this writes to `/kaggle/working/outputs/`, at the top level, so the bundle is
unambiguously part of the notebook Output rather than buried in the git clone. Download that
folder and commit it under `DENOISING_DIFFUSION/results/`.


In [ ]:
from src.evaluation.collect_outputs import collect_outputs

# Checkpoints are NOT listed: section 9 already puts them at the top of /kaggle/working,
# and this collector writes into /kaggle/working/outputs -- naming them would duplicate
# nine models inside the same Output.
_run_dir = collect_outputs(
    '05-unet-line-emission',
    [
        'nb05_seed_repeats.csv',                 # section 4: PSNR per config per seed
        'nb05_moment_by_seed.csv',               # section 6: moments per checkpoint
        'moment_all_arms_vs_v12.png',            # section 6d: the headline figure
        'moment_map_holdout_summary_sweepwinner.csv',
        'artifact_diagnostics_sweepwinner.csv',
        'artifact_diagnostics_sweepwinner.png',
        'hallucination_worst_channels.png',      # section 8
        'line_emission_unet_comparison.png',     # section 6c
        'sweepwinner_loss.png',                  # section 5
        'moments_*.png',                         # section 6b, one per holdout cube
    ],
    extra={'configs': sorted(CONFIGS), 'seeds': SEEDS, 'best': f'{BEST_CONFIG} seed {BEST_SEED}',
           'target_size': TARGET_SIZE, 'n_samples': N_SAMPLES,
           'metric': 'signal-masked + noise-clipped (moment_improvement)'},
)
print('\nAnything listed as NOT FOUND above did not get written this run.')